# 007 - JupyterLab Sidebar Chatbot 데모

JupyterLab **우측 사이드바**에 뜨는 챗봇입니다. 두뇌는 **langgraph 그래프(deepagents) + Claude** 이고, 설치부터 실행까지 **전부 노트북 셀 안에서** 끝납니다(jupyter 서버 재시작 불필요).

| 단계 | 하는 일 | jupyter 재시작? |
|---|---|---|
| ① 프론트 설치 | 셀에서 `%pip install <wheel>` → 브라우저 새로고침 → 우측 💬 탭 등장 | ❌ 불필요 |
| ② 두뇌 시작 | 셀에서 `start_graph_server()` (langgraph 그래프, localhost:8765) | ❌ 불필요 |
| ③ 대화 | 💬 탭에서 입력 → Claude(deepagents) 응답 | — |

> ⚠️ **온라인/개발 전용**: 두뇌가 Claude API(외부 네트워크)를 호출하고 anthropic/langchain-anthropic(httpx) 를 쓰므로 폐쇄망 배포용이 아닙니다. 키는 `ANTHROPIC_API_KEY` 환경변수로만 주입(노트북에 키 금지).
>
> ⚠️ 전제: 브라우저의 `127.0.0.1` 과 이 커널이 **같은 기계**여야 합니다(로컬/컨테이너/SSH 포트포워딩). single-file `.py` 가 아니라 빌드가 필요한 익스텐션입니다 — 자세한 건 `README.md` 참고.

## ① 프론트엔드 설치 — 노트북 셀에서 (jupyter 재시작 불필요)

터미널을 못 여는 환경이라면 **셀에서 직접 `%pip install`** 하면 됩니다. `%pip` 매직은 지금 이 커널(=jupyter 서버와 같은 env)에 설치하므로, labextension 자산이 `share/jupyter/labextensions/` 에 들어갑니다. 폐쇄망에서는 반입한 `.whl` 경로를 지정하세요.

In [ ]:
# 빌드 산출물 dist/ 의 wheel 을 설치 (`pip wheel . -w dist` 결과). 폐쇄망에선 반입한 .whl 경로로 바꾸세요.
%pip install dist/jlab_sidebar_chatbot-0.1.0-py3-none-any.whl
# 설치 후 → 브라우저에서 JupyterLab 페이지를 '새로고침'(Cmd/Ctrl+R) 하면 우측에 💬 탭이 생깁니다.
# (서버 재시작 불필요: jupyter 는 페이지를 그릴 때마다 labextensions 폴더를 다시 스캔하기 때문)

## ② 두뇌 서버 시작 — 이 셀 한 줄

커널 안에서 localhost HTTP 서버(기본 포트 8765)를 백그라운드로 띄웁니다. 우측 💬 탭에서 보낸 메시지가 이 서버로 전달됩니다. (방금 `%pip install` 한 패키지는 이 커널에서 바로 import 됩니다 — 커널 재시작도 불필요.)

In [ ]:
# 두뇌 = langgraph 그래프(deepagents + InMemorySaver). 키는 jupyter 서버 env 에서 상속.
from jlab_sidebar_chatbot import start_graph_server

start_graph_server()   # → http://127.0.0.1:8765 (백그라운드 스레드, ANTHROPIC_API_KEY 필요)
# 이제 오른쪽 사이드바의 💬 탭에서 Claude(deepagents)와 대화하세요.
# 중지:  from jlab_sidebar_chatbot import stop_graph_server; stop_graph_server()

## ③ 모델·시스템 프롬프트 바꾸기

두뇌는 **langgraph 그래프 하나**(deepagents 가 생성)입니다. 직접 만든 어댑터 클래스는 없습니다. 모델이나 시스템 프롬프트를 바꾸려면 그래프를 그렇게 만들어 주입하세요.

```python
from jlab_sidebar_chatbot import build_chat_graph, start_graph_server

graph = build_chat_graph(
    model="claude-sonnet-4-6",              # 또는 ANTHROPIC_MODEL 환경변수
    system_prompt="너는 SQL 전문가야. 쿼리 위주로 답해.",
)
start_graph_server(graph=graph)
```

- 멀티턴은 langgraph 체크포인터(`InMemorySaver`) + `thread_id`(=세션) 가 관리합니다.
- `tools=[...]` 등 deepagents 기능 확장도 `build_chat_graph(tools=...)` 로 가능합니다.

> ⚠️ 두뇌가 Claude API(외부 네트워크)를 호출하므로 **온라인/개발 전용**입니다. 키는 `ANTHROPIC_API_KEY` 환경변수로만 주입하세요(노트북 셀에 키를 적지 마세요).

## ④ 정리 — 설치한 익스텐션 삭제

데모가 끝나면 아래 셀로 두뇌 서버를 멈추고 패키지(프론트 labextension 포함)를 제거합니다. **jupyter 재시작 없이** 브라우저 새로고침만 하면 우측 💬 탭이 사라집니다.

In [ ]:
# (정리) 데모가 끝나면 두뇌 서버를 멈추고, 설치했던 익스텐션 패키지를 제거합니다.
from jlab_sidebar_chatbot import stop_graph_server
stop_graph_server()                       # 8765 두뇌 서버 중지 (먼저 — uninstall 전에)

%pip uninstall -y jlab_sidebar_chatbot    # 프론트 labextension 자산 포함 패키지 삭제
# → 브라우저를 '새로고침'하면 우측 💬 탭이 사라집니다 (jupyter 서버 재시작 불필요)